# 01 — Exploratory Data Analysis
### Winding Process Anomaly Detection Project
---
**Goal:** Understand the structure of normal vs anomalous winding cycles before building any detection model.  
**Dataset:** 500 winding cycles — 400 normal, 100 anomalous (5 anomaly types).


In [ ]:
import os
os.chdir(os.path.dirname(os.path.abspath('01_EDA.ipynb')))
print("Working directory:", os.getcwd())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--',
    'font.size':11,'axes.titlesize':12,
})
os.makedirs('../results/plots', exist_ok=True)
SEED = 42
print("Imports OK.")


## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/raw/winding_anomaly_data.csv')
print(f"Shape: {df.shape}")
print(f"Normal cycles  : {(df['is_anomaly']==0).sum()}")
print(f"Anomaly cycles : {(df['is_anomaly']==1).sum()}")
print(f"Anomaly rate   : {df['is_anomaly'].mean()*100:.1f}%")
print()
print(df.dtypes.to_string())


In [ ]:
df.head(5)


In [ ]:
print("Descriptive statistics — normal cycles:")
print(df[df['is_anomaly']==0].describe().round(2).to_string())


In [ ]:
print("Descriptive statistics — anomaly cycles:")
print(df[df['is_anomaly']==1].describe().round(2).to_string())


## 2. Anomaly Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Anomaly Overview', fontsize=14, fontweight='500')

# Class balance
ax = axes[0]
counts = df['is_anomaly'].value_counts().sort_index()
bars = ax.bar(['Normal', 'Anomaly'], counts.values,
              color=['#1D9E75','#E24B4A'], edgecolor='white', width=0.45)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontsize=10)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts.values)*1.25)

# Anomaly types
ax = axes[1]
atype_counts = df[df['is_anomaly']==1]['anomaly_type'].value_counts()
colors_a = ['#E24B4A','#BA7517','#378ADD','#7F77DD','#1D9E75']
bars = ax.barh(atype_counts.index, atype_counts.values,
               color=colors_a, edgecolor='white')
ax.set_title('Anomaly Types')
ax.set_xlabel('Count')
for bar, val in zip(bars, atype_counts.values):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
            str(val), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/plots/01_anomaly_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_anomaly_overview.png")


## 3. Signal Distributions — Normal vs Anomaly

In [ ]:
SIGNAL_FEATURES = [
    'tension_mean_N','tension_std_N','tension_peak_N',
    'power_mean_W','power_std_W',
    'temp_max_C','temp_gradient',
    'winding_speed_rpm'
]

normal  = df[df['is_anomaly']==0]
anomaly = df[df['is_anomaly']==1]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Signal Distributions — Normal vs Anomaly', fontsize=14, fontweight='500')
axes = axes.flatten()

for i, feat in enumerate(SIGNAL_FEATURES):
    ax = axes[i]
    ax.hist(normal[feat],  bins=30, alpha=0.65, color='#1D9E75',
            edgecolor='white', label='Normal',  density=True)
    ax.hist(anomaly[feat], bins=30, alpha=0.65, color='#E24B4A',
            edgecolor='white', label='Anomaly', density=True)
    ax.set_title(feat.replace('_',' '))
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

    # KS test
    ks_stat, ks_p = stats.ks_2samp(normal[feat], anomaly[feat])
    color = '#D85A30' if ks_p < 0.05 else '#888780'
    ax.text(0.97, 0.95, f'KS p={ks_p:.3f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color=color)

plt.tight_layout()
plt.savefig('../results/plots/02_signal_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_signal_distributions.png")


## 4. Anomaly Type Profiles — Radar / Heatmap

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_vis = StandardScaler()
df_scaled = df.copy()
df_scaled[SIGNAL_FEATURES] = scaler_vis.fit_transform(df[SIGNAL_FEATURES])

group_means = df_scaled.groupby('anomaly_type')[SIGNAL_FEATURES].mean()

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(group_means, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'shrink':0.7, 'label':'z-score vs overall mean'})
ax.set_title('Mean Signal z-scores by Anomaly Type\n(red = above normal mean, blue = below)', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('../results/plots/03_anomaly_type_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_anomaly_type_heatmap.png")


## 5. Correlation Heatmap — Normal Cycles Only

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = normal[SIGNAL_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r',
            vmin=-1, vmax=1, center=0,
            annot=True, fmt='.2f', linewidths=0.4,
            cbar_kws={'shrink':0.7})
ax.set_title('Correlation Matrix — Normal Cycles Only', fontsize=13)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('../results/plots/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 04_correlation_heatmap.png")


## 6. Boxplots — Key Signals by Anomaly Type

In [ ]:
key_features = ['tension_std_N', 'tension_peak_N', 'power_std_W', 'temp_max_C']
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle('Key Signal Distributions by Anomaly Type', fontsize=13)

palette = {
    'normal':'#1D9E75','tension_spike':'#E24B4A',
    'overheating':'#BA7517','power_surge':'#378ADD',
    'speed_instability':'#7F77DD','combined_stress':'#D85A30'
}

for ax, feat in zip(axes, key_features):
    order = ['normal','tension_spike','overheating','power_surge',
             'speed_instability','combined_stress']
    data_groups = [df[df['anomaly_type']==t][feat].values for t in order]
    bp = ax.boxplot(data_groups, patch_artist=True, widths=0.5,
                    medianprops={'color':'white','linewidth':2})
    for patch, t in zip(bp['boxes'], order):
        patch.set_facecolor(palette[t])
        patch.set_alpha(0.85)
    ax.set_xticklabels([t.replace('_','
') for t in order], fontsize=7)
    ax.set_title(feat.replace('_',' '))
    ax.set_ylabel(feat)

plt.tight_layout()
plt.savefig('../results/plots/05_boxplots_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 05_boxplots_by_type.png")


## 7. KS Test — Which Signals Differ Most Between Normal and Anomaly

In [ ]:
from scipy.stats import ks_2samp

print("Kolmogorov-Smirnov test — Normal vs Anomaly")
print(f"{'Feature':<25} {'KS statistic':>14} {'p-value':>10} {'Significant?':>14}")
print("-" * 65)

ks_results = []
for feat in SIGNAL_FEATURES:
    stat, p = ks_2samp(normal[feat], anomaly[feat])
    ks_results.append({'feature': feat, 'ks_stat': stat, 'p_value': p})
    sig = 'YES ***' if p < 0.05 else 'no'
    print(f"  {feat:<23} {stat:>14.4f} {p:>10.4f} {sig:>14}")

ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)


## 8. EDA Summary

In [ ]:
print("=" * 55)
print("EDA SUMMARY")
print("=" * 55)
print(f"  Total cycles       : {len(df)}")
print(f"  Normal cycles      : {(df['is_anomaly']==0).sum()}")
print(f"  Anomaly cycles     : {(df['is_anomaly']==1).sum()}")
print(f"  Anomaly rate       : {df['is_anomaly'].mean()*100:.1f}%")
print(f"  Anomaly types      : 5")
print(f"  Signal features    : {len(SIGNAL_FEATURES)}")
print(f"  Missing values     : {df.isna().sum().sum()}")
print()
print("KEY FINDINGS:")
print("  - tension_std_N and tension_peak_N most distinct for tension_spike")
print("  - temp_max_C and temp_gradient most distinct for overheating")
print("  - power_std_W most distinct for power_surge and speed_instability")
print("  - combined_stress anomalies show moderate deviations across all signals")
print("  - KS test confirms all 8 signals are statistically different (p<0.05)")
print()
print("NEXT STEP → 02_AnomalyDetection.ipynb")
